# Orbit Wars — RL Modeli ve Eğitim Notları

Bu notebook, repoda güncel olan **PlanetOnly Set Transformer + PPO + delayed reward** yığınının teorik temellerini ve pratik ayrıntılarını toplu halde açıklar. Odak, kod gezisi değil; **neden böyle tasarlandığının matematiksel ve RL teorisindeki karşılığıdır.**

İlgili dosyalar:

- `model.py` — full Set Transformer policy/value ağı.
- `model_planet_only.py` — fleet token kullanmayan sadeleştirilmiş varyant (şu an default).
- `obs_builder.py` — gözlem vektörü ve edge bias üretimi.
- `rl/ppo.py` — clipped surrogate PPO güncellemesi.
- `rl/buffer.py` — on-policy rollout buffer, GAE, `add_delayed_reward`.
- `rl/rewards.py` — reward konfigürasyonu ve step shaping.
- `rl/rollout.py` — self-play episode toplayıcı, delayed reward dağıtımı.
- `rl/production_reward_tracker.py`, `rl/fleet_tracker.py`, `rl/defense_tracker.py`, `rl/planet_loss_tracker.py` — delayed credit assignment mekanizmaları.

Notebook çalıştırılabilir değildir; amacı kuramsal bir referans olmaktır. Bazı kısa kod blokları, kod tabanına bir-bire-bir bağlantı kurmak için eklenmiştir.

## 1. Problem: Self-Play Zero-Sum MDP

Orbit Wars, iki (veya 4) oyunculu, ayrık zamanlı, kısmen gözlemlenebilir ama her oyuncuya simetrik bilgi veren bir strateji oyunudur. Eğitim açısından onu aşağıdaki gibi formalize ediyoruz:

- Durum: $s_t$ — environment’ın tam durumu (gezegenler, filolar, üretim, komet vs).
- Oyuncu $i$ için gözlem: $o_t^{(i)} = \phi_i(s_t)$ — kendi perspektifine göre normalize edilmiş feature tensörleri (`obs_builder.build_observation`).
- Oyuncu $i$ için aksiyon: $a_t^{(i)} = (\text{src}, \text{tgt}, \text{ship\_bin}, \text{is\_noop})$.
- Geçiş: $s_{t+1} = T(s_t, a_t^{(0)}, a_t^{(1)}, \dots)$ — tüm oyuncu aksiyonları eşzamanlı uygulanır.
- Ödül: $r_t^{(i)}$ — terminal galibiyet/mağlubiyet + dense shaping + delayed credit.

**Self-play**: tek bir politika $\pi_\theta$ her iki tarafta da oynatılır. Bu, **non-stationary ama simetrik** bir ortama sebep olur; çünkü $\pi$ güncellendikçe rakibin davranışı da değişir. PPO’nun on-policy özelliği ve küçük güncelleme adımı (clip), bu kaymayı tolere etmek için tasarımın tutarlı kısmıdır.

Amaç: Oyuncunun kendi ödül toplamının beklentisini maksimize etmek.

$$
J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[ \sum_{t=0}^{T} \gamma^{\,t}\, r_t \right].
$$

Zero-sum yapısı nedeniyle bir oyuncunun değeri, diğerinin değerinin yaklaşık olarak negatifidir; terminal ödülde $+1/-1$ simetrisi ile bu açıkça korunur (`normalize_terminal_rewards`).

## 2. Gözlem (Observation) Tasarımı

`obs_builder.py` env durumunu sabit şekilli numpy tensörlerine çevirir. Bu tensörler Set Transformer’in girdisidir. Temel ilkeler:

1. **Permutation invariance**: gezegen ve filo slotları set olarak sunulur, maske ile padding ayırt edilir.
2. **Kutupsal simetri**: oyunun merkezinde güneş var; özellikler $(r,\theta)$ tabanlı çıkarılır, $(\cos\theta, \sin\theta)$ ile döngüsellik korunur.
3. **Geometrik ön bilgi**: modelin uzaklık/açı öğrenmesi için **edge bias** tensörü hazırlanır ve attention’a eklenir.
4. **Aksiyon maskeleme**: yalnızca yeterli gemisi olan kendi gezegenlerimiz kaynak olabilir (`source_mask`).

### 2.1 Planet features (`PLANET_FEATURE_DIM = 31`)

Her gezegen için aşağıdaki özellikler üretilir (indeksler `obs_builder` içindeki `PF_IDX_*` sabitleriyle eşleşir):

- Sahiplik: `is_self`, `is_enemy`, `is_neutral` (one-hot).
- Ekonomi: normalize edilmiş gemi sayısı, üretim, yarıçap, `ships_to_cap`, `log_ships`.
- Konum: $r/50$, $\cos\theta$, $\sin\theta$, türler (static / rotating / comet), $\omega$.
- **Incoming fleet özeti (planet-only modelde kritik)**: kendi ve rakip filolarının ETA bucket’ları (`[0,5], (5,15], (15,30], 30+`), toplam gelen kuvvet, minimum ETA, en büyük düşman filosunun büyüklük/ETA’sı ve `net_threat = enemy_total - ships - self_total` gibi aggregate tehdit sinyalleri.

Bu zenginleştirme sayesinde planet-only model, fleet token’ları olmadan da savunma/saldırı zamanlamasını kavrayabilir. Eski 23-boyutlu checkpoint’ler `load_checkpoint` içinde state_dict’ten dinamik olarak tespit edilir (`planet_proj.weight.shape[1]`).

### 2.2 Fleet features

`f_max` slotluk filo tensörü üretilir (`FLEET_FEATURE_DIM = 11`): sahiplik, gemi sayısı, kaynak/hedef kodları, kalan ETA, kutupsal konum vb. PlanetOnly variant’ta `f_max = 0` olduğu için model bu tensörü kullanmaz; `full` variant’ta her filo ayrı token olarak kodlanır.

### 2.3 Global features

`GLOBAL_FEATURE_DIM = 12` boyutlu vektör: oyun turu, sahip olduğum/rakibin gezegen sayısı, toplam gemi oranı, pozisyonsal özet ve harcanmış süre gibi CLS-token benzeri global özetler içerir. Bu tek token, transformer’ın en sonuna eklenir ve `value head`’in girdisidir.

### 2.4 Edge bias

Her iki token arasında 5 boyutlu bir edge özelliği hesaplanır:

$$
e_{ij} = (d_{ij}/d_{\max},\; \Delta x,\; \Delta y,\; \sin\Delta\varphi_{\text{sun}},\; \cos\Delta\varphi_{\text{sun}}).
$$

Aynı yörünge hızına sahip iki gezegen için $\Delta\varphi_{\text{sun}}$ **zaman-değişmez** bir büyüklüktür; bu da modelin yörünge ilişkilerini öğrenmesini kolaylaştırır. Edge bias, **head başına additive attention bias** olarak projekte edilir:

$$
\text{scores} = \frac{QK^\top}{\sqrt{d}} + B_{\text{edge}}\;, \quad B_{\text{edge}} \in \mathbb{R}^{H\times T\times T}.
$$

Bu, modelin koordinatlardan mesafe yeniden öğrenmesini beklemek yerine geometriyi doğrudan önsel bilgi olarak vermek demek.

## 3. Model: Set Transformer Policy / Value

`OrbitWarsPolicy` (full) ve `PlanetOnlyOrbitWarsPolicy` aynı çekirdeği paylaşır. Token düzeni:

- **Full**: `[planet_0, ..., planet_{P-1}, fleet_0, ..., fleet_{F-1}, GLOBAL]`, yani $T = P_{\max} + F_{\max} + 1$.
- **PlanetOnly**: `[planet_0, ..., planet_{P-1}, GLOBAL]`, yani $T = P_{\max} + 1$.

Her tokena tip embedding eklenir (planet/fleet/global). Bu, heterojen tokenları $d_\text{model}$ uzayında ayırt etmeyi kolaylaştırır.

### 3.1 Biased multi-head attention

Standart çok kafalı dikkat mekanizmasına ek olarak, her pozisyon çifti için additive bias:

$$
\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}} + B_{\text{edge}} + M_{\text{pad}}\right) V,
$$

- $B_{\text{edge}}$: yukarıdaki geometrik bias.
- $M_{\text{pad}}$: padded key’ler için $-\infty$ maskesi.

Padded satırlarda softmax NaN üretir; `torch.nan_to_num` ile güvenli sıfırla doldurulur.

### 3.2 Pre-norm Transformer blokları

$$
x \leftarrow x + \text{Attn}(\text{LN}(x))\;,\quad x \leftarrow x + \text{FFN}(\text{LN}(x)).
$$

Varsayılan: `d_model=128`, `n_heads=4`, `n_layers=4`, FFN genişleme çarpanı 4 ve GELU aktivasyon. Bu ölçek, Orbit Wars gibi küçük küme uzayları için yeterince ifadeli, aynı zamanda CPU’da smoke test yapılabilecek kadar küçüktür.

### 3.3 Policy head’leri ve güncel rollout aksiyon semantiği

Modelin `forward` çıktısı üç policy head + value head üretir:

- **`src_logits`**: `(B, P_max + 1)`. Son slot klasik `act(...)` API’sinde global NO-OP slotudur.
- **`tgt_logits`**: `(B, P_max, P_max)`. Her source slot için target pointer dağılımı.
- **`ship_logits`**: `(B, P_max, n_ship_bins)`. Her source slot için gemi oranı dağılımı.
- **`value`**: `(B,)`. Global token’dan critic tahmini.

Ancak **training rollout şu anda `policy.act(...)` değil, `policy.act_source(...)` kullanır**. Bu önemli fark şudur:

```text
for src_slot in current_source_slots:
    launch/no-launch ~ Cat([0, src_raw[src_slot]])
    if launch:
        tgt      ~ Cat(tgt_logits[src_slot])
        ship_bin ~ Cat(ship_logits[src_slot])
```

Yani rollout içinde source seçimi global bir `src ~ Cat(P+1)` değil; owned source gezegenleri sırayla gezilir ve her source için ayrı bir **launch vs no-launch** kararı verilir.

Bu durumda tek source kararının log-prob’u:

$$
\log \pi(a_i|s_i, src_i) = \log p(launch_i|s_i,src_i)
+ \mathbb{1}[launch_i]\left(\log p(tgt_i|s_i,src_i)+\log p(ship_i|s_i,src_i)\right).
$$

PPO tarafında aynı semantik `evaluate_source_actions(...)` ile tekrar hesaplanır. Stored batch içindeki `src`, `tgt`, `ship_bin`, `is_noop`, `old_log_prob` alanları bu source-conditioned aksiyon semantiğine göredir.

### 3.4 Angle nasıl hesaplanıyor?

Policy **açı üretmez**. Policy sadece slot/ship kararı verir:

```text
src_slot, tgt_slot, ship_bin, is_noop
```

`obs_builder.decode_action_with_launch(...)` bunu env formatına çevirir. Gerçek açı `navigation.compute_launch(src, tgt, ships, planets, angular_velocity)` tarafından hesaplanır:

```text
[from_planet_id, angle_in_radians, ships]
```

`compute_launch` hareketli hedef için intercept ETA hesaplar, `atan2(intercept_y - src.y, intercept_x - src.x)` ile radyan açı üretir, güneş ve diğer gezegen çarpışma kontrollerinden geçmeyen rotaları reddeder.

### 3.5 Neden iki model variant’ı var?

- **Full**: filo trajektorileri token olarak verilir. `T = P_max + F_max + 1`.
- **PlanetOnly (default)**: `f_max = 0`; fleet token yoktur. Fleet bilgisi planet feature içindeki incoming ETA bucket aggregate’leri ve global feature’lar üzerinden gelir. Saved batch içinde `fleet_feats` alanı kalır ama şekli `(N, 0, 11)` olur.

Model seçimi `train.py` içindeki `_make_policy` ve `model.py`/`model_planet_only.py` dosyalarında yapılır. `build_observation(..., planet_feature_dim=policy.planet_dim, f_max=policy.f_max)` ile checkpoint mimarisi observation boyutuyla eşleştirilir.

### 3.6 Güncel mimari snapshot ve bilinen sınırlar

Bu bölüm, kod tabanındaki güncel neural network mimarisini kısa ve denetlenebilir şekilde özetler.

#### Encoder

Default eğitim yolu `PlanetOnlyOrbitWarsPolicy` kullanır:

```text
planet_feats: (B, P_max, 31)
global_feats: (B, 12)
edge_bias   : (B, P_max + 1, P_max + 1, 5)
T           : P_max + 1
```

`f_max=0` olduğu için tekil fleet tokenları yoktur. Filo bilgisi planet feature içindeki incoming ETA bucket aggregate'larına ve global feature'lara sıkıştırılır. Bu, token sayısını küçültür ve PPO training'i daha stabil yapar; karşılığında tek tek fleet trajectory bilgisini modele doğrudan vermez.

#### Actor head'leri

Güncel actor üç parça üretir:

```text
src_logits  = Linear(planet_tok) + global/no-op slot

tgt_logits[src, tgt]
            = (W_q planet_tok[src]) · (W_k planet_tok[tgt]) / sqrt(d_model)

ship_logits[src]
            = Linear(planet_tok[src])
```

Burada önemli mimari gerçek şudur: **target seçimi source-target pairwise pointer skorudur**, ama **ship bin head şu an target-conditioned değildir**. `ship_logits` sadece source token embedding'inden gelir; target'ın gemi sayısı, production değeri, neutral/enemy olması veya rotating fırsat durumu ship bin dağılımına ancak source embedding'in attention üzerinden taşıdığı dolaylı bilgilerle yansır.

Bu nedenle gözlenen şu davranış mimari olarak şaşırtıcı değildir:

```text
Agent doğru target'a saldırmayı öğrenebilir,
ama bazen o target için yetersiz gemi bin'i seçebilir.
```

Bunu mimariyle güçlendirmek istersek olası bir sonraki adım:

```text
ship_logits[src, tgt] = MLP([src_tok, tgt_tok, edge(src,tgt), launch_features(src,tgt)])
```

Yani ship head'i source-target pair-conditioned yapılabilir.

#### Rotation bilgisi modelde var mı?

Evet. Planet feature içinde rotating bilgisi açıkça vardır:

```text
is_static, is_rotating, is_comet, omega, r_sun, cos(theta), sin(theta)
```

Ayrıca edge bias içinde güneş merkezli faz farkı bulunur:

```text
sin(phi_j - phi_i), cos(phi_j - phi_i)
```

Bu yüzden model bir target'ın rotating olduğunu ve source-target orbital ilişkisini görebilir. Ancak şu bilgi **explicit** değildir:

```text
bu source-target çifti için şu an launch valid mi?
ETA kaç?
kaç turn sonra açı/fırsat kapanacak?
kaç turn sonra tekrar reachable olacak?
```

Bu bilgiler şu anda policy input'una feature olarak verilmez; launch feasibility `decode_action_with_launch -> navigation.compute_launch` aşamasında deterministik olarak kontrol edilir. Invalid trajectory varsa action env'e gönderilmez ve reward/invalid-action mekanizması üzerinden öğrenilir.

#### Macro-agent ilkesi

Policy angle üretmez ve üretmemelidir. Aksiyon uzayı macro düzeydedir:

```text
hangi kaynaktan?
hangi hedefe?
ne kadar gemiyle?
launch mı no-op mu?
```

Açı, intercept ETA, spawn offset, sun collision ve planet sweep collision `navigation.py` tarafından deterministik çözülür. Son navigation düzeltmeleriyle intercept hesabı env fiziğine uyumludur: fleet spawn offset'i (`src.radius + 0.1`) hesaba katılır ve source planet sweep collision kontrol edilir.

#### Kısa mimari risk listesi

- `ship_head` target-conditioned değil; yetersiz neutral/enemy saldırılarına katkı verebilir.
- Future opportunity/reachability feature yok; rotating gezegenlerde fırsat penceresi tamamen learned representation'a kalır.
- Planet-only model tekil fleet tokenlarını görmez; incoming aggregate bucket'ları yeterli olmadığı durumlarda hassas savunma/saldırı zamanlaması zorlaşabilir.
- Reward attribution doğru olsa bile source başına aynı turn içinde çok sayıda transition üretilir; global step reward bu transition'lara paylaştırılır.

In [ ]:
# Mimari hızlı özet (çalıştırmak için opsiyonel)
from model_planet_only import PlanetOnlyOrbitWarsPolicy
from obs_builder import PLANET_FEATURE_DIM, P_MAX_DEFAULT

policy = PlanetOnlyOrbitWarsPolicy(d_model=128, n_heads=4, n_layers=4)
total_params = sum(p.numel() for p in policy.parameters())
print(f"Variant: {policy.model_variant}")
print(f"Token count T = {policy.p_max + 1}, planet_dim = {policy.planet_dim}")
print(f"Parametre sayısı: {total_params:,}")


## 4. Öğrenme Algoritması: PPO + GAE

Politika güncellemesi **Proximal Policy Optimization (clip variant)** ile yapılır (`rl/ppo.py`).

### 4.1 Clipped surrogate objective

$r_t(\theta) = \pi_\theta(a_t|s_t) / \pi_{\theta_{\text{old}}}(a_t|s_t)$ olmak üzere:

$$
L^{\text{CLIP}}(\theta) = \mathbb{E}_t\!\left[\min\!\left(r_t\hat A_t,\; \text{clip}(r_t, 1-\epsilon, 1+\epsilon) \hat A_t\right)\right].
$$

Bu terim, politika dağılımının **bir adımda çok fazla kaymasını** yapısal olarak sınırlar.

### 4.2 PPO batch içinde hangi action log-prob tekrar hesaplanıyor?

Rollout `act_source(...)` kullandığı için PPO update de klasik `evaluate_actions(...)` değil, `evaluate_source_actions(...)` çağırır:

```text
out = policy(batch observation)
ev = policy.evaluate_source_actions(
    out,
    src=batch["src"],
    tgt=batch["tgt"],
    ship_bin=batch["ship_bin"],
    is_noop=batch["is_noop"],
)
new_logp = ev["log_prob"]
ratio = exp(new_logp - old_log_prob)
```

Bu kontrol kritik: stored `old_log_prob` ile update sırasındaki `new_logp` aynı source-conditioned faktörizasyonu kullanmalıdır. Aksi halde PPO ratio yanlış olur.

### 4.3 Value loss (opsiyonel clipping)

$$
L^{VF}(\theta) = \frac{1}{2}\max\left((V_\theta - R)^2,\; (V^{\text{clip}} - R)^2\right),
$$

burada $V^{\text{clip}} = V_{\theta_{\text{old}}} + \text{clip}(V_\theta - V_{\theta_{\text{old}}}, -\epsilon_V, \epsilon_V)$.

### 4.4 Entropi bonusu

Toplam kayıp:

$$
L(\theta) = L^{\text{CLIP}} + c_1 L^{VF} - c_2 H[\pi_\theta](s).
$$

Güncel `PPOConfig` default değerleri: `value_coef=0.5`, `entropy_coef=0.05`, `clip_range=0.2`, `max_grad_norm=0.5`.

### 4.5 Generalized Advantage Estimation

GAE-$\lambda$ yüksek varyanslı Monte Carlo getirisi ile düşük varyanslı TD(0) arasında interpolasyon sağlar:

$$
\delta_t = r_t + \gamma V(s_{t+1})\,\mathbb{1}[\neg\text{done}_t] - V(s_t),\qquad
\hat A_t = \sum_{l \geq 0} (\gamma\lambda)^l \delta_{t+l}.
$$

`rl/buffer.py` içinde her oyuncunun transition’ları için ayrı rekürsif hesaplama yapılır (`compute_gae`). Default: `gamma=0.99`, `lam=0.95`.

### 4.6 Advantage normalizasyonu

`iter_minibatches(...)` içinde advantage’lar tüm rollout buffer üzerinde ortalama=0, std=1’e normalize edilir. Saved game batch dosyalarında ise `batch_from_indices(..., normalize_advantages=False)` kullanıldığı için ham GAE advantage saklanır.

## 5. Ödül (Reward) Sistemi — Güncel Kod Durumu

Bu bölüm model incelemesi için reward sinyalinin PPO batch’e nasıl girdiğini açıklar. Burada tasarım önerisi değil, mevcut kod akışı anlatılır.

### 5.1 Terminal ödül

`normalize_terminal_rewards(raw_rewards, draw_value=...)` terminal env reward’unu normalize eder:

- Tek kazanan varsa: kazanan `+1`, diğerleri `-1`.
- Tek kazanan yoksa: herkes `draw_value` alır.

`production_simple` presetinde güncel `draw_terminal_reward=-1.0` olarak ayarlıdır. Bu değer CLI’dan `--draw-terminal-reward` ile override edilebilir.

### 5.2 Step shaping

`compute_step_reward(...)` şu bileşenleri hesaplayabilir:

- capture/denial production delta,
- ship diff,
- ship loss,
- invalid action,
- step cost,
- terminal reward.

`production_simple` presetinde step-level capture/ship shaping kapalıdır (`capture_value_weight=0`, `denial_value_weight=0`, `ship_diff_weight=0`, `ship_loss_weight=0`). Bu yüzden ana sinyal launch transition’larına delayed/immediate olarak eklenen tracker reward’larından gelir.

### 5.3 Launch transition reward kaynakları

Bir source için transition buffer’a eklendikten sonra reward şu yollardan değişebilir:

```text
immediate:
  invalid action penalty
  airborne cost
  source defense deficit penalty
  terminal reward share if terminal step

delayed:
  fleet combat/capture outcome
  production capture outcome
  defense hold outcome for NO-OP holds
```

`RolloutBuffer.add_reward(idx, value)` aynı transition’ın reward alanını artırır/azaltır. `add_delayed_reward(idx, reward, event_turn=t, discount=d)` ise event gecikmesine göre `reward * d**delay` ekler.

### 5.4 production_simple formülleri

Güncel preset:

```text
reward_scale = 0.001
fleet_airborne_weight = 1.0
fleet_kill_weight = 1.0
fleet_waste_weight = 1.0
planet_loss_weight = 1.0
delayed_reward_discount = 1.0
```

Capture reward:

$$
R_{capture}= production \cdot (T_{max}-t) \cdot opponent\_multiplier \cdot ship\_share \cdot reward\_scale
$$

Fleet combat reward:

$$
R_{combat}= \frac{enemy\_ships\_killed}{my\_total\_ships\_at\_launch}\cdot reward\_scale
$$

Combatta hedefe ulaşan ama capture edemeyen filolar için `own_waste` cezası kaldırılmıştır; sadece `destroyed_enroute` durumunda waste negatif yazılır.

Airborne cost:

$$
R_{airborne}= - fleet\_airborne\_weight \cdot ships\_sent \cdot ETA \cdot risk\_multiplier \cdot reward\_scale
$$

Source defense deficit penalty:

$$
R_{deficit}= -\max(0, enemy\_incoming - ships\_remaining\_after\_launch) \cdot planet\_loss\_weight \cdot reward\_scale
$$

### 5.5 Son diagnostic bulgusu

Son incelemede reward/PPO kodu değiştirilmeden şu ölçüm yapıldı:

- Checkpoint: `production_simple_v7_softening/policy_iter00009.pt`
- Aynı rollout batch üzerinde launch vs no-op reward/advantage karşılaştırıldı.
- PPO `evaluate_source_actions(...)` ile stored `old_log_prob` semantiği uyumlu şekilde tekrar hesaplandı.

Ölçüm sonucu:

```text
base:
  reward launch mean ≈ -0.433
  reward noop mean   ≈ +0.000
  normalized advantage launch mean ≈ -0.168
  normalized advantage noop mean   ≈ +0.063

fleet_airborne_weight=0 ablation:
  reward launch mean ≈ +0.027
  reward noop mean   ≈ +0.000
  normalized advantage launch mean ≈ +0.065
  normalized advantage noop mean   ≈ -0.024
```

Aynı batch üzerinde tek PPO update denemesi:

```text
before mean p_launch ≈ 0.241
after  mean p_launch ≈ 0.202
delta launch logit mean ≈ -0.259
```

Bu, mevcut batch etiketlerinin launch aksiyonlarını ortalamada azaltma yönünde gradient ürettiğini gösterir. Bu hücre sadece mevcut modeli/batch’i incelemek için kaydedilmiştir; reward tasarım kararını otomatik olarak değiştirmez.

## 6. Rollout ve Training Pipeline

Bir epoch (iteration) içinde şu sıra izlenir (`rl/train.py` + `rl/rollout.py`):

1. **Self-play rollout**: tek bir policy her iki tarafta oynatılır.
   - Her oyuncu için `build_observation` ile planet/edge tensörleri üretilir.
   - `policy.act_source` ile aksiyon örneklenir; log-prob ve value saklanır.
   - Transition buffer’a eklenirken `turn=t` de kaydedilir; böylece gelecekteki delayed ödüller doğru step’e bağlanır.
   - Fleet launch’ları ilgili trackerlara **buffer_idx** ile kayıt edilir.
   - Env.step’ten sonra: fleet arrival, defense hold, production capture/loss, planet loss olaylarından gelen ödüller `add_delayed_reward` ile buffer’a yazılır.
2. **GAE**: episode sonu bootstrap $V=0$ ile `compute_gae` çalıştırılır; her oyuncu ayrı ayrı rekürsif olarak işlenir.
3. **PPO update**: $K$ epoch boyunca shuffled minibatchler üzerinde clipped surrogate loss.
4. **Logging/checkpoint**: KL, clip fraction, value loss, entropi, grad norms ve update ratio metrikleri JSONL’e yazılır, `latest.pt` ve `policy_iter*.pt` kaydedilir.

### 6.1 İki-oyunculu buffer

Her env step’inde iki transition eklenir (p0 ve p1). GAE hesaplanırken `compute_gae` oyuncuları ayırıp her biri için ayrı rekürsyon yapar; böylece iki oyuncunun getirileri birbirine karışmaz.

### 6.2 Seed yönetimi

`seed = train_cfg.seed * 100000 + it * 1000 + ep` formülü, iterasyon ve episode ikilileri için deterministik ama çeşitli seedler üretir; aynı başlangıç konfigürasyonuyla eğitim tekrar üretilebilir.

## 7. Hiperparametreler ve Teşhis Notları

Varsayılan `TrainConfig` ve `PPOConfig` (`rl/train.py`, `rl/ppo.py`):

| Hiperparametre | Default | Sezgi |
|---|---|---|
| `gamma` | 0.99 | Uzun-vadeli plan: ~100 adımlık horizon. |
| `gae_lambda` | 0.95 | Yüksek MC / düşük TD arasında tipik PPO dengesi. |
| `learning_rate` | 3e-4 | Adam ile Transformer için başlangıç değeri. |
| `clip_range` | 0.2 | Politika güncelleme büyüklüğü sınırı. |
| `value_clip_range` | 0.2 | Critic aşırı güncellemesini engeller. |
| `ppo_epochs` | 4 | Aynı buffer üzerinden 4 tur geçiş. |
| `batch_size` | 256 | Minibatch boyutu. |
| `entropy_coef` | 0.05 | Exploration bonusu. |
| `max_grad_norm` | 0.5 | Gradient clipping. |
| `episodes_per_iter` | 1 | Her oyundan sonra update. |
| `max_episode_steps` | 500 | Env’in doğal episode sınırı. |
| `model_variant` | `planet_only` | Fleet token yok; fleet bilgisi planet aggregate feature içinde. |

### 7.1 KL ve clip fraction teşhisi

- `approx_kl` çok büyükse: learning rate veya `clip_range` fazla olabilir.
- `clip_fraction` yüksekse: PPO çoğu örneği kesiyor olabilir.
- `entropy` hızla 0’a inerse: policy deterministikleşiyor; launch/no-op distribution da dahil olmak üzere exploration bitiyor olabilir.
- `value_loss` sürekli artıyorsa veya çok büyük başlıyorsa: reward scale / return scale kontrol edilmeli.

### 7.2 Launch/no-op collapse teşhisi için batch metrikleri

Sadece `launch/p` loguna bakmak yeterli değildir. Bir rollout batch üzerinde şu ayrımlara bakılmalı:

```text
is_noop == False:
  reward mean / return mean / advantage mean
is_noop == True:
  reward mean / return mean / advantage mean
```

Eğer normalize advantage ortalaması launch için negatif, no-op için pozitifse PPO update launch logitlerini düşürür. Bu davranış PPO hatası değildir; batch’in policy’ye verdiği etikettir.

### 7.3 Game batch kaydetme

`train.py` artık her oyun için batch kaydedebilir:

```bash
--save-game-batches --batch-save-dir checkpoints/game_batches
```

Her `.pt` dosyasında:

```text
iteration, episode, seed
transition_start, transition_end, num_transitions
episode_stats
train_cfg, ppo_cfg, reward_cfg
batch
```

`batch` alanında:

```text
planet_feats, fleet_feats, global_feats, edge_bias
planet_mask, fleet_mask, source_mask
src, tgt, ship_bin, is_noop, player
old_log_prob, old_value, returns, advantages
```

Planet-only modelde `fleet_feats` ve `fleet_mask` boş kalır:

```text
fleet_feats: (N, 0, 11)
fleet_mask : (N, 0)
```

Saved batch içinde `advantages` ham GAE değeridir; PPO training sırasında `iter_minibatches` bunu buffer genelinde normalize eder.

## 8. Çalıştırma: Komutlar

### 8.1 production_simple (önerilen)

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --model-variant planet_only \
  --device cuda \
  --iterations 1000 \
  --episodes-per-iter 1 \
  --checkpoint-dir checkpoints/production_simple \
  --log-path logs/production_simple.jsonl
```

Bu komut, normalize edilmiş `production_simple` preset’ini kullanır. `--delayed-reward-discount` flag’ine ihtiyaç yoktur; default `1.0`.

### 8.2 CPU smoke test

```bash
orbit_wars_venv/bin/python -m rl.train \
  --reward-system production_simple \
  --model-variant planet_only \
  --device cpu \
  --iterations 10 \
  --episodes-per-iter 1 \
  --checkpoint-dir checkpoints/smoke \
  --log-path logs/smoke.jsonl
```

İlk birkaç epoch’ta `valL` ~10–500 ve `H` ~1.5–3 görmen beklenir. Daha büyük `valL` (≥1000) veya `H ≈ 0` reward kalibrasyonunda sorun olduğunu gösterir.

### 8.3 Resume

```bash
orbit_wars_venv/bin/python -m rl.train \
  --resume-from checkpoints/production_simple/latest.pt \
  --checkpoint-dir checkpoints/production_simple \
  --log-path logs/production_simple.jsonl \
  --reward-system production_simple --model-variant planet_only \
  --device cuda --iterations 2000 --episodes-per-iter 1
```

### 8.4 Replay izleme (watch)

```bash
orbit_wars_venv/bin/python -m rl.watch \
  --ckpt checkpoints/production_simple/latest.pt \
  --html replays/production_simple.html \
  --device cuda \
  --deterministic
```

Bu, `rl/watch.py` üzerinden Kaggle native HTML replay üretir. Browser'da açmak için:

```bash
xdg-open replays/production_simple.html
```

Stokastik izlemek için `--deterministic` bayrağı kaldırılır.

## 9. Test Matrisi ve Sanity Checks

`tests/test_rl.py` başlıca şunları kanıtlar:

- `RolloutBuffer.add_delayed_reward` `reward * discount**delay` uyguluyor mu?
- `compute_gae` oyuncuların transition’larını ayrı işliyor mu?
- `act_source` ile üretilen log-prob, `evaluate_source_actions` ile tekrar hesaplanınca eşleşiyor mu? (`tests/test_model.py`)
- `production_simple` capture/combat/defense-deficit formülleri beklenen ölçeği veriyor mu?
- `normalize_terminal_rewards(..., draw_value=...)` draw değerini doğru yayıyor mu?
- `production_simple` preset’i güncel `reward_scale=0.001`, `fleet_airborne_weight=1.0`, `draw_terminal_reward=-1.0`, `delayed_reward_discount=1.0` değerlerini taşıyor mu?

`tests/test_obs_builder.py`:

- `PLANET_FEATURE_DIM = 31`.
- Incoming bucket ve aggregate threat alanları beklenen indekslerde.
- `decode_action_with_launch` hareketli hedef için `compute_launch` ETA bilgisini expose ediyor.

`tests/test_model.py`:

- Forward shape’leri.
- `act_source` source’u sabit tutuyor ve `evaluate_source_actions` log-prob eşleşiyor.

## 10. Kısa Özet

- **Observation**: permutation-invariant set; `PLANET_FEATURE_DIM=31`; geometrik edge bias; incoming fleet özetleri planet feature içinde.
- **Model**: biased Set Transformer; default `planet_only`; `f_max=0`; actor source-conditioned `act_source` rollout semantiği kullanıyor.
- **Actor heads**: target seçimi source-target pointer skoru; `ship_head` şu an sadece source-conditioned ve target-conditioned değil.
- **Rotation**: `is_rotating`, `omega`, `r_sun`, `cos(theta)`, `sin(theta)` ve orbital edge bias modelde var; ama future reachability/opportunity window explicit feature değil.
- **Action decode**: policy açı üretmez; `compute_launch` env için `[from_id, angle_in_radians, ships]` üretir ve spawn offset + collision kontrollerini deterministic çözer.
- **Öğrenme**: clipped PPO + GAE; PPO update `evaluate_source_actions` ile stored action log-prob’unu tekrar hesaplar.
- **Batch**: `RolloutBuffer` transition → GAE → normalized minibatch; isteğe bağlı `--save-game-batches` ile her oyun `.pt` olarak saklanır.
- **Reward diagnostic**: son ölçümde launch transition advantage’ları no-op’a göre daha negatif göründü; bu, PPO’nun launch logitlerini düşürmesini açıklıyor. Bu notebook bulguyu kaydeder, otomatik reward tasarımı kararı vermez.

Daha derin okuma için: `docs/reward_system.md`, `docs/reward.md`, `docs/production_simple_reward.md`, `docs/math.md`.